In [1]:
import os
import math
from datetime import datetime

import torch
from torchvision import transforms, datasets

In [ ]:
DATASET_DIR = "~/Desktop/Datasets/PKLot/PKLotSegmented"

### Data Split Protocol
- UFPR04 and UFPR05 for training and validation
    - 70% (first days) days for training and 30% (last days)
- PUCPR for test

In [2]:
transform = transforms.Compose([
    transforms.Resize(128),
    transforms.PILToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [3]:
subsets = ["UFPR04", "UFPR05"]

all_dates = []
for subset in subsets:
    climatic_condition = os.listdir(f"{DATASET_DIR}/{subset}")
    for weather in climatic_condition:
        dates = os.listdir(f"{DATASET_DIR}/{subset}/{weather}")
        temp = [(datetime.strptime(date, '%Y-%m-%d').date(), f"{DATASET_DIR}/{subset}/{weather}/{date}") for date in dates]

        all_dates.extend(temp)

all_dates.sort(key=lambda x: x[0])

train_days = math.ceil(len(all_dates) * 0.7)
train = all_dates[:train_days]
validation = all_dates[train_days:]

In [4]:
train_ds = []
for date in train:
    _, path = date
    train_ds.append(datasets.ImageFolder(path, transform))

final_train = torch.utils.data.ConcatDataset(train_ds)

validation_ds = []
for date in validation:
    _, path = date
    validation_ds.append(datasets.ImageFolder(path, transform))

final_validation = torch.utils.data.ConcatDataset(validation_ds)

In [5]:
root_dir = F"{DATASET_DIR}/PUC"

test_days = 0

test_ds = []
climatic_condition = os.listdir(f"{root_dir}")
for weather in climatic_condition:
    dates = os.listdir(f"{root_dir}/{weather}")
    for date in dates:
        path = f"{root_dir}/{weather}/{date}"
        test_ds.append(datasets.ImageFolder(path, transform))
    
    train_days += len(dates)

final_test = torch.utils.data.ConcatDataset(test_ds)

In [6]:
print(f"Total train days quantity: {len(train)}")
print(f"Total validation days quantity: {len(validation)}")
print(f"Total test days quantity: {train_days}")
print("------------------------------------------------------------------------")
print(f"Train dataset size: {len(final_train)}")
print(f"validation dataset size: {len(final_validation)}")
print(f"test dataset size: {len(final_test)}")

Total train days quantity: 71
Total validation days quantity: 30
Total test days quantity: 115
------------------------------------------------------------------------
Train dataset size: 174817
validation dataset size: 96811
test dataset size: 424223
